# HW1｜學務規章檢索

**個人作業　W3 出題 → W5 繳交**

---

## 這次作業要做什麼

我們在課堂上談過：一個 RAG 系統回答得好不好，關鍵往往不在模型，而在**檢索這一步有沒有把對的東西找出來**。
找回來的是雜訊，後面的模型再強也救不回來。

也談過另一件事——把文件切成小塊的時候，如果只是按固定字數硬切，
很容易把原本連在一起的內容切斷：條文和它的但書分家、表格的規則跟它管的項目跑到不同塊去。

**這次作業，你要用輔大資管的真實規章，親自驗證第二件事是不是真的。**

做法很單純：同一批文件、同一組問題，只換切塊的方式，看命中率差多少。

我刻意把「生成回答」這一段拿掉了。原因是，當你看到一個 RAG 系統答得爛，
最直覺的反應是去調提示詞或換模型——但問題多半出在檢索。
這次只做檢索，你就只剩檢索可以調，才會真的去面對切塊這件事。

---

## 這份 notebook 怎麼用

**看到 ✍️ 開頭的格子，就是你要作答的地方。** 雙擊那一格就能開始打字，把表格的空欄位填滿。

**標有 `TODO` 的程式碼格要你補完。** 每個 TODO 只需要改一行，那一行標了 `← 只改這一行`。
沒改就執行，會跳出中文訊息告訴你卡在哪。

**其他程式碼格直接執行就好**，不用改。

---

## 三件明確不做的事

| | 為什麼 |
|---|---|
| **不做生成** | 輸出只能是「條文片段 ＋ 出處 ＋ 分數」的清單。若你的程式印出「根據規定，碩士班需修滿 36 學分」這種句子，就超出範圍了 |
| **不做多輪檢索** | 一個問題、一次檢索、一組結果。不可以拿第一次的結果去產生新問題再查一次 |
| **不用向量資料庫** | 這個規模用不到。若你堅持要用，必須額外說明它內部怎麼算相似度、top-k 怎麼排序 |

---

## 評分重點

| 項目 | 配分 |
|---|---|
| 可運作 | 20 |
| 實驗設計 | 25 |
| **失敗診斷** | **30** |
| 回望誠實度 | 25 |

配分最重的是**失敗診斷**——說得出「哪裡失敗、為什麼失敗」，比命中率高更重要。

---

## 繳交前檢查

1. 從頭到尾「重新啟動並全部執行」一次，確認不會出錯
2. 所有 ✍️ 格都有內容
3. 檔名改為 `HW1_學號_姓名.ipynb`

> ⚠️ **先開 GPU**：執行階段 → 變更執行階段類型 → T4 GPU

---
# 區塊一　假設設定

1-3 編號在後，但**時間上最先完成**。猜測必須在你動手之前寫下，否則區塊五就沒有對照的基準。

## ✍️ 1-3　事前猜測

> **⏰ W3 課堂最後 10 分鐘當場完成並繳交，之後不得修改。**
>
> 你認為哪些因素會影響「能不能找到正確條文」？至少寫三項，依影響力由大到小排序。

一時想不出來的話，從下面幾個方向去想。
**但要寫出具體的因素，不要把方向名稱直接抄下來當答案。**

| 想的方向 | 可以問自己 |
|---|---|
| 文件本身 | 這些規章長什麼樣子？裡面有表格嗎？同一件事會不會分散在好幾份文件裡？ |
| 切塊方式 | 一塊該多大？該切在哪裡？切斷了會發生什麼事？ |
| 問題本身 | 是不是有些問題天生就比較難查？難在哪裡？ |
| 比對方式 | 電腦怎麼判斷「這段文字跟這個問題有關」？它擅長什麼、不擅長什麼？ |

| 排序 | 我認為會影響的因素 | 理由（一句話） |
|---|---|---|
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |

## ✍️ 1-1　名詞定義

> 用一句話分別定義。

這五個名詞課堂上都講過，投影片裡查得到——**但請用你自己的話寫**。
照抄投影片或網路上的定義不會拿到分數，因為那看不出你懂不懂。

判斷標準很簡單：把你的定義拿給沒修過這門課的同學看，他看得懂嗎？

| 名詞 | 我的定義 |
|---|---|
| Token |  |
| Embedding |  |
| 餘弦相似度 |  |
| Chunking |  |
| 稀疏檢索與稠密檢索的差別 |  |

## ✍️ 1-2　流程說明

> 一個問題進來，到系統回傳條文片段，中間經過哪些步驟？**每一步的輸入與輸出各是什麼？**

**去哪裡找答案**：往下捲到區塊四，那幾格程式碼就是這個流程的實作。
從 `load_docs()` 開始，一路看到 `search()`，每一格都在做流程中的一步。

看懂它在做什麼，然後用自己的話描述——**不要照抄函式名稱當答案**。
「呼叫 `build_index()`」不算描述；
「把每一塊文字轉換成一串數字，用這串數字代表它的意思」才算。

列數不夠可以自己加。

| 步驟 | 這一步在做什麼 | 輸入 | 輸出 |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |
| 4 |  |  |  |

---
# 環境設定

以下三格直接執行。每次 Colab 重新啟動都要重跑一次（約 1–2 分鐘）。

In [ ]:
# 安裝套件
!pip install -q sentence-transformers
!apt-get install -qq poppler-utils

print("套件安裝完成")

# 預期輸出：套件安裝完成

In [ ]:
# 下載語料。全班路徑一致為 /content/hw-corpus
REPO = "https://github.com/____/fju-im-agent-hw.git"

!rm -rf /content/repo /content/hw-corpus
!git clone --depth 1 -q $REPO /content/repo
!cp -r /content/repo/hw-corpus /content/hw-corpus

print("語料下載完成")

# 預期輸出：語料下載完成

In [ ]:
# 確認語料
import glob, os

pdfs = sorted(glob.glob("/content/hw-corpus/01-系級/*.pdf"))
txts = sorted(glob.glob("/content/hw-corpus/text/01-系級/*.txt"))

print(f"原始 PDF：{len(pdfs)} 份")
print(f"參考解析：{len(txts)} 份")
print()
for f in txts[:3]:
    print("  ", os.path.basename(f))
print("   ...")

# 預期輸出：原始 PDF 19 份、參考解析 19 份

本次作業使用的是**系所層級**的規章：修業規則 6 個年度版、必選修科目表 4 版、碩士班課表 9 份。

- `01-系級/` —— 原始 PDF
- `text/01-系級/` —— **參考解析輸出**，也就是教師事先把這些 PDF 轉成文字的成果

參考解析的用途有二：讓全班的實驗有共同的起點；以及萬一你的解析卡住，仍然做得下去。

---
# 區塊二　素材整備

**這一區不要求你寫出完美的表格抽取**——找出問題在哪就夠了，力氣留給區塊四。

## 2-1　重現基準解析

參考解析是用 `pdftotext` 這個命令列工具產生的，但**參數要你自己找出來**。

目標：讓你的輸出與 `text/01-系級/` 裡的檔案**完全一致**。

> 提示：在程式碼格裡執行 `!pdftotext -h` 可以看到所有參數說明。
> 你需要兩個——一個讓表格的欄位間距保留下來，一個指定 UTF-8 編碼。
>
> 這一步在練**可重現性**：別人給你一個結果，你要能自己跑出同樣的東西。

In [ ]:
# ══════════════════ TODO 1 ══════════════════
# 填入 pdftotext 的參數，讓輸出與參考解析完全一致
#
# 提示：先執行 !pdftotext -h 看說明
#       需要兩個參數：保留欄位間距、指定 UTF-8 編碼
# ═══════════════════════════════════════════

PDFTOTEXT_ARGS = ""      # ← 只改這一行，例如 "-aaa -bbb XXX"

if not PDFTOTEXT_ARGS:
    raise NotImplementedError("請先完成 TODO 1：填入 pdftotext 的參數")

In [ ]:
# 用你填的參數跑一次，並與參考解析逐字比對
SRC = "/content/hw-corpus/01-系級/必選修科目表-114資管碩-中英對照.pdf"
OUT = "/content/my_parse.txt"

!pdftotext $PDFTOTEXT_ARGS "$SRC" "$OUT"

ref = open("/content/hw-corpus/text/01-系級/必選修科目表-114資管碩-中英對照.txt",
           encoding="utf-8").read()
mine = open(OUT, encoding="utf-8").read()

print("完全一致" if ref == mine else "不一致，請調整參數再試")
print(f"參考 {len(ref)} 字 / 我的 {len(mine)} 字")

# 預期輸出：完全一致（參數對的話）

## ✍️ 2-1　填答

| 問題 | 你的回答 |
|---|---|
| 我用了哪些參數？ |  |
| 輸出與參考解析一致嗎？ |  |
| 若不一致，差在哪裡？ |  |
| 你認為原因是什麼？ |  |

## 2-2　損壞檢視

把解析出來的文字，跟原始 PDF 擺在一起看。

**建議同時開啟** `/content/hw-corpus/01-系級/必選修科目表-114資管碩-中英對照.pdf`
（左側檔案面板可以下載或預覽），對照下一格印出來的文字。

這份是**表格型 PDF**，最容易看出解析的問題。

In [ ]:
# 把參考解析的一段印出來，與原始 PDF 對照
path = "/content/hw-corpus/text/01-系級/必選修科目表-114資管碩-中英對照.txt"
text = open(path, encoding="utf-8").read()

lines = text.split("\n")
for i, line in enumerate(lines[28:52], start=29):
    print(f"{i:>3} | {line}")

# 想看別的段落，把 28:52 換成其他範圍

## ✍️ 2-2　損壞檢視

> 對照原始 PDF，**至少找出三處損壞**。
>
> 可以觀察：表格欄位之間的對應關係、直排的標籤文字、跨行被拆開的備註、中英文黏在一起。

| # | 損壞在哪 | 具體描述（原本應該是什麼、變成了什麼） |
|---|---|---|
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |

---
# 區塊三　判準確立

沒有判準就無法談「檢索好不好」，只能憑感覺。下面這 10 題就是你在區塊四要**逐題送進檢索器**的問題。

In [ ]:
# 評測集（這格已經寫好，直接執行，內容不要改）
#   source：正確答案所在的檔案，檔名須包含這個字串
#   keys  ：該段落必須同時出現的關鍵字，用來判定是否命中

EVAL = [
    {"id": 1,  "type": "直查",   "q": "碩士班畢業至少須修滿多少學分？",
     "source": "系修業規則", "keys": ["畢業學分數", "36"]},
    {"id": 2,  "type": "直查",   "q": "碩士班前二學年每學期選課的上限與下限各為多少學分？",
     "source": "系修業規則", "keys": ["選課上限", "18"]},
    {"id": 3,  "type": "表格",   "q": "人工智慧學群「5 選 3」的必選課程是哪五門？",
     "source": "必選修科目表", "keys": ["巨量資料探勘"]},
    {"id": 4,  "type": "表格",   "q": "「資訊管理講座」的授課語言有何特殊規定？",
     "source": "必選修科目表", "keys": ["資訊管理講座"]},
    {"id": 5,  "type": "條件",   "q": "碩士班學生大學部未修過「系統分析與設計」，該如何處理？",
     "source": "系修業規則", "keys": ["系統分析與設計", "敏捷式軟體開發"]},
    {"id": 6,  "type": "條件",   "q": "碩士班學生若入學前未通過中高級英文檢定，須做什麼？",
     "source": "系修業規則", "keys": ["碩一結束前"]},
    {"id": 7,  "type": "層級",   "q": "碩士班的英文檢定標準是什麼？具體分數對應為何？",
     "source": "系修業規則", "keys": ["TOEIC"]},
    {"id": 8,  "type": "層級",   "q": "碩士班須修畢幾門「以英語授課的專業課程」？",
     "source": "系修業規則", "keys": ["碩士班", "以英語授課的專業課程"]},
    {"id": 9,  "type": "跨版本", "q": "大學部專業必修學分數在哪一學年度變動？變成多少？",
     "source": "系修業規則", "keys": ["專業必修課程", "64"]},
    {"id": 10, "type": "無答案", "q": "碩士班修業年限最長幾年？",
     "source": None, "keys": None},
]

for e in EVAL:
    print(f"{e['id']:>2}. [{e['type']}] {e['q']}")

# 預期輸出：10 行題目

### 兩件要注意的事

**第 10 題不計入命中率。** 它的答案不在本次語料裡，改為觀察題——
向量檢索一定會回傳結果，即使正確答案根本不存在。等一下你會親眼看到。

**第 9 題是半自動判定。** 程式只檢查你有沒有找到「64 學分」那一條；
「哪一學年度變動」需要你自己比對不同年度的版本，寫在填答區。

## ✍️ 3-1　題型辨識

> 每一題屬於哪一型，上面已經標好了。你要說明的是**判斷的依據**：
> 這一題需要從幾份文件、幾個位置取得資訊？

| # | 需要幾份文件／幾個位置 | 我的判斷依據 |
|---|---|---|
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |
| 4 |  |  |
| 5 |  |  |
| 6 |  |  |
| 7 |  |  |
| 8 |  |  |
| 9 |  |  |
| 10 |  |  |

## ✍️ 3-2　失敗預測

> 結合 2-2 你看到的損壞，預測哪幾題會失敗。**區塊五要回頭核對這個預測。**

| 我預測會失敗的題號 | 理由 |
|---|---|
|  |  |
|  |  |
|  |  |

---
# 區塊四　實作驗證

## 兩種切塊策略

對應課堂上提過的兩種做法——依文件本身的結構切，或依固定的字數切：

| | 做法 |
|---|---|
| **固定長度** | 每 500 字一塊，相鄰塊重疊 100 字 |
| **依結構切** | 以「第 X 條」為邊界切分修業規則 |

**500／100 這組參數全班統一，不要自己改。** 理由跟你做實驗時一樣：
這次要比的是「切法」，如果字數也跟著變，差異就混進了參數的影響，比較就不乾淨了。

## 兩次跑，只動一個變因

| 跑 | 解析來源 | 切塊策略 |
|---|---|---|
| 一 | 參考解析 | 固定長度 500／100 |
| 二 | 參考解析 | 依結構切 |

In [ ]:
# 載入全部參考解析（系級 19 份）
import glob, os

def load_docs():
    docs = []
    for path in sorted(glob.glob("/content/hw-corpus/text/01-系級/*.txt")):
        docs.append({"file": os.path.basename(path),
                     "text": open(path, encoding="utf-8").read()})
    return docs

DOCS = load_docs()
print(f"載入 {len(DOCS)} 份文件，共 {sum(len(d['text']) for d in DOCS):,} 字")

# 預期輸出：載入 19 份文件，共 130,586 字

In [ ]:
# 切塊策略 A：固定長度（這格已經寫好，直接執行）
def chunk_fixed(docs, size=500, overlap=100):
    """每 size 字切一塊，相鄰塊重疊 overlap 字。"""
    chunks = []
    for d in docs:
        t = d["text"]
        step = size - overlap
        for i in range(0, len(t), step):
            piece = t[i:i + size]
            if piece.strip():
                chunks.append({"file": d["file"], "text": piece})
    return chunks

CH_FIXED = chunk_fixed(DOCS)
print(f"固定長度切法：{len(CH_FIXED)} 塊")

# 預期輸出：固定長度切法：339 塊

## TODO 2　依結構切

下一格要你完成 `split_one()`：把一份文件切成多塊，回傳字串清單。

> **提示 1**　修業規則的條號長這樣：第一條、第六條、第十一條（中文數字）。
>
> **提示 2**　可以用 `re.split` 搭配 lookahead，切的時候讓「第X條」留在後一段的開頭。
> 例如 `re.split(r"(?=第[一二三四五六七八九十]+條)", text)`。
>
> **提示 3**　必選修科目表和課表**沒有**「第X條」，這個切法對它們切不開。
> 那該怎麼辦？整份當成一塊、還是改成按行切？**你的選擇要能在 5-1 說明理由。**

In [ ]:
# ══════════════════ TODO 2 ══════════════════
# 完成 split_one()：把一份文件的文字切成多塊，回傳字串清單
# ═══════════════════════════════════════════

import re

def split_one(text):
    pieces = None      # ← 只改這一行

    if pieces is None:
        raise NotImplementedError("請先完成 TODO 2：把文件切成多塊")
    return pieces


def chunk_structured(docs):
    chunks = []
    for d in docs:
        for piece in split_one(d["text"]):
            if piece.strip():
                chunks.append({"file": d["file"], "text": piece})
    return chunks

CH_STRUCT = chunk_structured(DOCS)
print(f"依結構切法：{len(CH_STRUCT)} 塊")

# 預期輸出：塊數會依你的實作而不同，大致落在 100～400 之間

In [ ]:
# 看看兩種切法切出來長什麼樣，感受一下差別
print("【固定長度】第 20 塊：")
print(CH_FIXED[20]["text"][:150].replace("\n", " "))
print()
print("【依結構切】第 20 塊：")
print(CH_STRUCT[20]["text"][:150].replace("\n", " "))

In [ ]:
# 建立索引：把每一塊文字轉成向量
from sentence_transformers import SentenceTransformer
import numpy as np

MODEL_NAME = "intfloat/multilingual-e5-small"   # 換模型前請先確認它支援中文

model = SentenceTransformer(MODEL_NAME)

def build_index(chunks):
    """e5 系列模型要求段落加上 'passage: ' 前綴，問題加 'query: '。"""
    texts = ["passage: " + c["text"] for c in chunks]
    emb = model.encode(texts, normalize_embeddings=True,
                       batch_size=64, show_progress_bar=True)
    return np.array(emb)

print("模型載入完成：", MODEL_NAME)

# 預期輸出：模型下載進度條，最後印出模型名稱

## TODO 3　算相似度

下一格要你補上「問題向量」與「所有文字塊向量」之間的相似度分數。

> **提示**　`build_index()` 用了 `normalize_embeddings=True`，
> 所以所有向量都是**單位長度**。
>
> 單位向量的餘弦相似度，等於它們的**內積**。
>
> `emb` 的形狀是 `(塊數, 維度)`，`q` 的形狀是 `(維度,)`。
> 想一想：怎麼一次算出「q 對每一塊」的分數，得到長度為「塊數」的陣列？

In [ ]:
# ══════════════════ TODO 3 ══════════════════
# 計算 query 與所有 chunk 的相似度分數
# ═══════════════════════════════════════════

def search(query, chunks, emb, k=5):
    q = model.encode("query: " + query, normalize_embeddings=True)

    scores = None      # ← 只改這一行

    if scores is None:
        raise NotImplementedError("請先完成 TODO 3：計算相似度分數")

    top = np.argsort(-scores)[:k]
    return [{"rank": r + 1, "score": float(scores[i]),
             "file": chunks[i]["file"], "text": chunks[i]["text"]}
            for r, i in enumerate(top)]

In [ ]:
# 命中判定（這格已經寫好，直接執行）
import re

def normalize(s):
    """壓掉所有空白。pdftotext 會用大量空格排版，不處理會影響字串比對。"""
    return re.sub(r"\s+", "", s)

def is_hit(result, item):
    """單一檢索結果是否命中：來源檔名對得上，且關鍵字全部出現。"""
    if item["source"] is None:
        return False
    if item["source"] not in result["file"]:
        return False
    body = normalize(result["text"])
    return all(normalize(kw) in body for kw in item["keys"])

## TODO 4　算命中率

下一格要你補上「這一題有沒有成功」的判斷。

> **Recall@k 的定義**：對每一題，只要 top-k 之中**至少有一個**結果命中，這題就算成功。
>
> `results` 是 `search()` 回傳的清單，`is_hit(r, item)` 判斷單一結果是否命中。
>
> 想一想：怎麼表達「清單裡至少有一個滿足條件」？

In [ ]:
# ══════════════════ TODO 4 ══════════════════
# 判斷這一題是否成功（top-k 至少一個命中）
# ═══════════════════════════════════════════

def evaluate(chunks, emb, k=5, verbose=True):
    scored, hit_ids, miss_ids = 0, [], []

    for item in EVAL:
        if item["source"] is None:      # 第 10 題不計分
            continue
        scored += 1
        results = search(item["q"], chunks, emb, k=k)

        hit = None      # ← 只改這一行

        if hit is None:
            raise NotImplementedError("請先完成 TODO 4：判斷是否命中")

        (hit_ids if hit else miss_ids).append(item["id"])
        if verbose:
            mark = "命中" if hit else "未中"
            print(f"{mark}  第{item['id']:>2}題 [{item['type']}] {item['q'][:22]}")

    recall = len(hit_ids) / scored
    print(f"\nRecall@{k} = {len(hit_ids)}/{scored} = {recall:.1%}")
    print(f"失敗題號：{miss_ids}")
    return {"k": k, "recall": recall, "hit": hit_ids, "miss": miss_ids}

## 跑一：固定長度切法（基準線）

In [ ]:
EMB_FIXED = build_index(CH_FIXED)

r3_fixed = evaluate(CH_FIXED, EMB_FIXED, k=3, verbose=False)
print()
r5_fixed = evaluate(CH_FIXED, EMB_FIXED, k=5, verbose=True)

# 預期輸出：兩組 Recall 數字與失敗題號

## ✍️ 跑一觀察

> **跑完立刻寫**，不要等兩次都跑完才回頭想，那時候細節已經忘了。

| 項目 | 結果 |
|---|---|
| Recall@3 |  |
| Recall@5 |  |
| 失敗題號 |  |
| 當下的觀察（一句話） |  |

## 跑二：依結構切法（對照）

In [ ]:
EMB_STRUCT = build_index(CH_STRUCT)

r3_struct = evaluate(CH_STRUCT, EMB_STRUCT, k=3, verbose=False)
print()
r5_struct = evaluate(CH_STRUCT, EMB_STRUCT, k=5, verbose=True)

## ✍️ 跑二觀察

| 項目 | 結果 |
|---|---|
| Recall@3 |  |
| Recall@5 |  |
| 失敗題號 |  |
| 當下的觀察（一句話） |  |

In [ ]:
# 兩次跑的對照表
import pandas as pd

pd.DataFrame([
    {"切法": "固定長度 500/100", "塊數": len(CH_FIXED),
     "Recall@3": f"{r3_fixed['recall']:.1%}", "Recall@5": f"{r5_fixed['recall']:.1%}"},
    {"切法": "依結構切", "塊數": len(CH_STRUCT),
     "Recall@3": f"{r3_struct['recall']:.1%}", "Recall@5": f"{r5_struct['recall']:.1%}"},
])

## 第 10 題：答案不存在的時候

第 10 題問「碩士班修業年限最長幾年」——**這個規定不在本次語料裡**（它在校級學則）。

但檢索器不知道這件事。看看它會怎麼做。

In [ ]:
q10 = EVAL[9]["q"]
print("問題：", q10)
print()

for r in search(q10, CH_STRUCT, EMB_STRUCT, k=3):
    print(f"[第{r['rank']}名] 分數 {r['score']:.4f}　來源：{r['file']}")
    print("   ", r["text"][:100].replace("\n", " "))
    print()

# 預期輸出：仍然回傳 3 筆結果，而且分數不見得比其他題低

## ✍️ 無答案題觀察

| 問題 | 你的回答 |
|---|---|
| 它回傳了什麼？ |  |
| 分數跟其他題比起來如何？ |  |
| **檢索器有辦法自己說「我不知道」嗎？為什麼？** |  |

## ✍️ 4-2　失敗歸因

> 把兩次跑失敗的題目逐一分類。選「其他」的話要具體說明。
>
> 分類：**切壞了／解析壞了／本來就沒答案／其他**

| 題號 | 哪一次跑失敗 | 分類 | 具體說明 |
|---|---|---|---|
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |

---
# 區塊五　結果評鑑

數據都跑完了，接下來要解釋它們。

## ✍️ 5-1　策略判斷

| 問題 | 你的回答 |
|---|---|
| 哪一種切法比較好？ |  |
| 你依據什麼判定？ |  |
| 哪一類題目差距最大？ |  |
| 哪一類幾乎沒差？ |  |
| TODO 2 裡你怎麼處理「沒有第X條」的文件？為什麼這樣選？ |  |

## ✍️ 5-2　預測核對

> 回去看 3-2 你寫的預測。

| 問題 | 你的回答 |
|---|---|
| 你猜中了哪幾題？ |  |
| 哪幾題沒猜中？ |  |
| 沒猜中的，實際原因是什麼？ |  |

## ✍️ 5-3　主張檢驗

> 課堂上提過：**如果只是按固定字數硬切，容易把原本連在一起的內容切斷，造成上下文斷裂。**

| 問題 | 你的回答 |
|---|---|
| 你的數據支持這個主張嗎？ |  |
| 最具說服力的證據是哪一題？ |  |
| 為什麼那一題最有說服力？ |  |

## ✍️ 5-4　回望

> 回去看 1-3 你在課堂上寫下的三項猜測。

| 你的猜測 | 猜對還是猜錯 | 錯在哪裡 |
|---|---|---|
| 猜測 1 |  |  |
| 猜測 2 |  |  |
| 猜測 3 |  |  |

**如果現在重寫 1-3，你會怎麼排序？為什麼？**

| 排序 | 因素 | 改變排序的理由 |
|---|---|---|
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |

---
# 加分項

兩項合計上限 10%，**不做不扣分**。

## 加分項一（5%）　改進解析

在基準解析之上做**一項**改進，用較好的那種切法再跑一次，報告命中率的變化與成因。
三類擇一：

| 類型 | 做法 |
|---|---|
| 換工具 | `pdfplumber`（有 `extract_table()`）、`PyMuPDF`、`pypdf` |
| 調參數 | 頁面範圍、編碼、其他排版相關參數 |
| 加後處理 | 針對 2-2 找到的損壞寫修正程式，例如把被拆成兩行的備註接回去 |

> **這一項評的是你能否解釋成因，不是命中率高低。**
> 改完反而變差也是有效結果，說得清楚就算數。

## 加分項二（5%）　證明檢索是瓶頸

接上生成，用它證明檢索是瓶頸：同一批問題，比較「好檢索」與「刻意調爛的檢索」
餵給同一個模型，回答品質差多少。

## 選作（不加分，可寫進心得）

改用 **BM25 稀疏檢索**跑一次。課堂上提過，稠密檢索「對數字與精確名詞不敏感」，
而這份評測集有大量數字題——看看那句話對不對。

In [ ]:
# 加分項的程式碼寫在這裡。不做可以留空。
#
# 加分項一的做法提示：
#   1. 用你選的方式重新解析，做成跟 DOCS 一樣的格式
#      docs2 = [{"file": 檔名, "text": 文字}, ...]
#   2. CH2  = chunk_structured(docs2)     # 或 chunk_fixed，用你認為較好的那種
#   3. EMB2 = build_index(CH2)
#   4. evaluate(CH2, EMB2, k=5)
#   5. 跟前面的結果比較，說明差異的成因

## ✍️ 加分項填答

不做可以留空。

| 問題 | 你的回答 |
|---|---|
| 我做了哪一項改進？ |  |
| 改進前後的 Recall@5 |  |
| 變好還是變差？ |  |
| **成因是什麼？** |  |

---
# 繳交前檢查

- [ ] 從頭到尾「重新啟動並全部執行」一次，沒有出錯
- [ ] 四個 TODO 都完成了
- [ ] 所有 ✍️ 格都有內容
- [ ] 檔名改為 `HW1_學號_姓名.ipynb`